# 01 — Full-Text, Vector, and Hybrid Search on the Same Query

**Module 1.** Determine whether and how Azure AI Search meets the retrieval
requirements of an enterprise generative AI solution.

**This notebook is a demonstration, not a build-along.** It runs against a
finished index so you can see what you're building toward. You build that
index yourself in Module 2, and it must exist before you run this notebook.


## Setup — run once per kernel session

In [1]:
# Connects to the index Module 2 built. Run once per kernel session.
import re
import sys
sys.path.insert(0, "..")

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

from config import load_config

config = load_config()
credential = DefaultAzureCredential()
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

print("Querying index:", config.search_index_name)
# Confirms which index you're querying. If this doesn't match .env, stop.


Querying index: state-driver-manuals


In [2]:
# Two helpers the cells below reuse. No output here.
def top_hits(query, state_filter=None, k=3):
    """Hybrid + semantic — the best retrieval this index can do. Returns the
    top k, not just the top 1: a single ~2000-character page chunk often
    bundles several unrelated facts (a page of road-sign definitions, say),
    so BM25/vector similarity alone doesn't reliably rank the one sentence
    that answers a pinpoint question first. Rather than quietly picking
    result 1 and risking an off-topic passage, this shows the top few so
    you can see which one actually answers the question."""
    return list(search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")],
        query_type="semantic",
        semantic_configuration_name="semantic-config",
        filter=f"state eq '{state_filter}'" if state_filter else None,
        top=k,
        select=["state", "title", "chunk"],
    ))


GARBLED_CHAR = "\ufffd"  # U+FFFD REPLACEMENT CHARACTER — PDF-encoding noise, not content


def show_results(hits, label="", answer_pattern=r"\d+\s*mph", preview_len=220):
    """Flags each result with a rule, not a judgment call: GARBLED means
    PDF-encoding noise (many U+FFFD characters) — skip this one. ANSWER
    means the chunk matches answer_pattern (a number next to "mph", by
    default) — this is the one to read. Neither flag changes the order
    results print in; it only labels the same ranking the index returned.

    Both checks run on the same preview text that gets printed, not the
    full ~2000-character chunk behind it — a chunk can carry garbled text
    (a page footer, say) well past what's shown, and flagging on the full
    chunk falsely marks a clean-looking preview as GARBLED."""
    if not hits:
        print(f"{label}: no results")
        return
    print(label)
    for hit in hits:
        full = hit["chunk"].strip().replace("\n", " ")
        preview = full[:preview_len]
        reranker = hit.get("@search.reranker_score")
        if preview.count(GARBLED_CHAR) > 3:
            flag = "  <-- GARBLED, do not read"
        elif re.search(answer_pattern, preview, re.IGNORECASE):
            flag = "  <-- READ THIS ONE"
        else:
            flag = ""
        print(f"  [{hit['state']} — {hit['title']}, reranker={reranker:.2f}]{flag}")
        print(f"    {preview}...")
    print()


def mode_table(query, key_pattern, label, state_filter=None, k=3, preview_len=90):
    """Top k passages for one query under each search mode, as readable text.

    key_pattern is a regex for the words that make a passage a real answer.
    A check mark means the passage contains it, so you can see why a mode
    won instead of trusting a state name. Ranks are shown, not scores:
    keyword (BM25) and vector scores use different scales."""
    flt = f"state eq '{state_filter}'" if state_filter else None
    vector_query = [VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")]
    select = ["state", "chunk"]
    modes = {
        "keyword": search_client.search(search_text=query, filter=flt, top=k, select=select),
        "vector": search_client.search(search_text=None, vector_queries=vector_query, filter=flt, top=k, select=select),
        "hybrid": search_client.search(search_text=query, vector_queries=vector_query, filter=flt, top=k, select=select),
    }
    summary = []
    for mode, results in modes.items():
        hits = list(results)
        matched = 0
        print(mode.upper())
        for rank, hit in enumerate(hits, 1):
            text = " ".join(hit["chunk"].replace(GARBLED_CHAR, "").split())
            found = re.search(key_pattern, text, re.IGNORECASE)
            matched += bool(found)
            start = max(0, found.start() - 30) if found else 0
            if start:
                start = text.find(" ", start) + 1 or start  # begin at a word boundary
            mark = "yes" if found else "no "
            print(f"  {rank}. {hit['state']:<11} {mark}  ...{text[start:start + preview_len]}...")
        summary.append(f"{mode} {matched}/{len(hits)}")
        print()
    print(f"Passages containing \"{label}\":  " + "   ".join(summary))


---
## Clip 1 (3 min) — The Retrieval Layer and Why Enterprises Need One

No cold-model comparison — a capable model may well know a school-zone
speed limit already, and that isn't the argument. The argument: you cannot
tell where its answer came from, cannot keep it current as your documents
change, and cannot confine it to content your organization approved.
Retrieval gives you all three, regardless of what the model already knows.

**Citation note:** the chunking in Module 2 doesn't preserve the source
PDF's original page numbers (that needs a layout-aware skill, out of scope
here) — cite by state + document instead. The
quoted passage is verbatim from the index, so a learner can still verify it
in three seconds without a page number.

**This prints the top 3 matches, not just 1.** A single
~2000-character page chunk can bundle several unrelated facts (a page of
road-sign definitions, say), so similarity search alone doesn't always rank
the one sentence that answers a pinpoint question first. Note which
rank your target fact lands on for each state, rather than assuming
result 1 is always it.
(This is exactly the problem Module 2 Clip 3's filtering and semantic
ranking exist to reduce — it's already wired in here.)

Two runs of the same query, same code, only the filter changes.


In [5]:
hero_question = "What's the speed limit in a school zone?"

unfiltered = top_hits(hero_question)
show_results(unfiltered, "Run 1 — unfiltered")


Run 1 — unfiltered
  [wisconsin — wisconsin-motorists-handbook.pdf, reranker=3.01]
    rain, snow and/or ice.  Speed limits:  Speed limits are the fastest speed you are allowed to drive in the area you are in. These signs are on  the side of most roads. It is illegal to go over the speed limit that is on t...
  [florida — florida-driver-handbook.pdf, reranker=3.01]  <-- GARBLED, do not read
    less you can see.        68 	 ©2023 Florida Department of Highway Safety and Motor Vehicles  	���������������������������������������������������������������������������������������������������������������  Right-of-Way ...
  [florida — florida-driver-handbook.pdf, reranker=2.97]  <-- READ THIS ONE
    the speed limit and may  receive a citation for speeding.   Florida Standard Speed Limits 	◆ School Zones: 20 MPH  	◆ Municipal, Business, or Residential  Area: 30 MPH*  	◆ Streets and Highways: 55 MPH*  	◆ Limited Acces...



In [ ]:
filtered = top_hits(hero_question, state_filter="florida")
show_results(filtered, "Run 2 — filter='state eq \'florida\''")

Run 2 — filter='state eq 'florida''
  [florida — florida-driver-handbook.pdf, reranker=3.01]  <-- GARBLED, do not read
    less you can see.        68 	 ©2023 Florida Department of Highway Safety and Motor Vehicles  	���������������������������������������������������������������������������������������������������������������  Right-of-Way ...
  [florida — florida-driver-handbook.pdf, reranker=2.97]  <-- READ THIS ONE
    the speed limit and may  receive a citation for speeding.   Florida Standard Speed Limits 	◆ School Zones: 20 MPH  	◆ Municipal, Business, or Residential  Area: 30 MPH*  	◆ Streets and Highways: 55 MPH*  	◆ Limited Acces...
  [florida — florida-driver-handbook.pdf, reranker=2.69]
    highlight the danger zone where road  users should not stop. These markings indicate the clearance needed  for trains to safely pass, as any object within the dynamic envelope  has the potential to be struck when the tra...



---
## Clip 2 (4 min) — Full-Text, Vector, and Hybrid Search on the Same Query

One function, three calls: keyword-only (`search_text=`), vector-only
(`vector_queries=`), and both together (hybrid). Two queries chosen so each
mode's failure mode is visible on screen.


In [6]:
# TLSAE is "Traffic Law and Substance Abuse Education", the course Florida
# requires before a first license. The acronym appears in exactly one of
# the five manuals (Florida), so keyword search can match it directly.
print("Query A — exact acronym:")
mode_table("What is TLSAE?", key_pattern=r"TLSAE", label="TLSAE")


Query A — exact acronym:


KEYWORD
  1. florida     yes  ...Substance Abuse Education (TLSAE) Course If you have never held—or do not have in your pos...
  2. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  3. florida     yes  ...Substance Abuse Education (TLSAE) Course Basic Driver Improvement (BDI) Course Advanced Dr...



VECTOR
  1. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  2. florida     yes  ...school are exempt from the TLSAE requirement Parental Consent for Minors If you are under ...
  3. california  no   ...are treated fairly, consistently with laws and regulations, and with dignity and respect. ...



HYBRID
  1. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  2. florida     yes  ...school are exempt from the TLSAE requirement Parental Consent for Minors If you are under ...
  3. california  no   ...are treated fairly, consistently with laws and regulations, and with dignity and respect. ...

Passages containing "TLSAE":  keyword 3/3   vector 2/3   hybrid 2/3


In [7]:
# The manuals say "following distance" or "the four-second rule". They never
# say "space in front of me", so keyword search has no phrase to match.
print("Query B — paraphrase, the manuals use different words:")
mode_table(
    "How much space should I leave in front of me?",
    key_pattern=r"following distance|second rule",
    label="following distance / second rule",
)


Query B — paraphrase, the manuals use different words:


KEYWORD
  1. virginia    no   ...you are backing up, have someone outside to help guide you. To back the trailer to the lef...
  2. florida     no   ...unless it is a path or part of a roadway reserved for bicycles. They must ride single file...
  3. california  no   ...whether you have enough space to pass whenever you approach: • An oncoming vehicle or bicy...



VECTOR
  1. wisconsin   yes  ...front: Use the “four second following distance rule” as explained previously when you are ...
  2. virginia    yes  ...the two-, three- and four-second rule to determine if you are following far enough behind ...
  3. virginia    yes  ...the two-, three- and four-second rule to determine if...



HYBRID
  1. wisconsin   yes  ...front: Use the “four second following distance rule” as explained previously when you are ...
  2. virginia    yes  ...the two-, three- and four-second rule to determine if you are following far enough behind ...
  3. virginia    yes  ...the two-, three- and four-second rule to determine if...

Passages containing "following distance / second rule":  keyword 0/3   vector 3/3   hybrid 3/3


---
## Clip 3 (3 min) — Getting Data In, and Where the Service Should Live

**No notebook code in this clip.** It happens in the Azure portal and on a
decision slide, because the objective is to *identify* connectors and
*compare* deployment models, not build either.

**Portal:** Azure AI Search service → **Import data** → look at the
connector list — Blob Storage, Azure SQL, Cosmos DB, SharePoint — and what
each is suited to. Don't configure one.

**Decision slide:** standalone Azure AI Search — reuse across apps, own the
lifecycle, full API control — versus inside a Microsoft Foundry project —
one project scope, wired to models, faster start.


---
## Done

Module 1 covers all three enabling objectives for Terminal Objective 1
using nothing but the index Module 2 already built. Next up: Module 2
shows how the retrieval layer underneath it works.
